In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard component imports
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl
import plotly.express as px

# Data and utility imports
import pandas as pd
import base64
import os

# Import CRUD Python module from Project One
from CRUD_Python_Module import AnimalShelter

# Allows Dash to work properly through Codio/Jupyter proxy
JupyterDash.infer_jupyter_proxy_config()


###########################
# Data Manipulation / Model
###########################

# Hardcoded MongoDB authentication for the aacuser account
username = "aacuser"
password = "SNHU1234"

# Connect to MongoDB through the CRUD module
db = AnimalShelter(username, password)


def get_dataframe_from_query(query):
    """Run a MongoDB query through the CRUD module and return a clean DataFrame."""

    records = db.read(query)
    dataframe = pd.DataFrame.from_records(records)

    # Return an empty dataframe with expected columns if no records are found
    if dataframe.empty:
        return dataframe

    # Remove MongoDB ObjectId because Dash cannot display it cleanly
    if "_id" in dataframe.columns:
        dataframe.drop(columns=["_id"], inplace=True)

    # Convert map coordinate columns to numeric values
    if "location_lat" in dataframe.columns:
        dataframe["location_lat"] = pd.to_numeric(dataframe["location_lat"], errors="coerce")

    if "location_long" in dataframe.columns:
        dataframe["location_long"] = pd.to_numeric(dataframe["location_long"], errors="coerce")

    return dataframe


# Load all AAC records for the initial dashboard state
df = get_dataframe_from_query({})

# Build a stable column list for the data table
table_columns = [
    {"name": column, "id": column, "deletable": False, "selectable": True}
    for column in df.columns
]


###########################
# Query / Controller Helpers
###########################

def build_rescue_query(filter_type):
    """Build MongoDB queries for each rescue filter option."""

    if filter_type == "water":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Labrador Retriever Mix|Chesapeake Bay Retriever|Newfoundland",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "mountain":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "disaster":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # Reset returns the full unfiltered dataset
    return {}


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Load Grazioso Salvare logo
logo_file = "Grazioso Salvare Logo.png"

if not os.path.exists(logo_file):
    logo_file = "Grazioso Salvare Logo(1).png"

encoded_image = base64.b64encode(open(logo_file, "rb").read()).decode()

app.layout = html.Div([

    # Branding area
    html.Div([
        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image),
                style={
                    "height": "120px",
                    "display": "block",
                    "margin": "auto"
                }
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),

        html.Center(html.B(html.H1("Grazioso Salvare Rescue Animal Dashboard"))),
        html.Center(html.H3("Created by Yosif Al-khazaali")),
        html.Center(html.P("CS 340 Project Two - Client/Server Development"))
    ]),

    html.Hr(),

    # Interactive filter options
    html.Div([
        html.H3("Interactive Rescue Type Filter"),

        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Reset", "value": "reset"},
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"}
            ],
            value="reset",
            labelStyle={
                "display": "inline-block",
                "margin-right": "25px",
                "font-weight": "bold"
            }
        )
    ]),

    html.Hr(),

    # Data table
    html.H3("Austin Animal Center Outcomes Data"),
    dash_table.DataTable(
        id="datatable-id",
        columns=table_columns,
        data=df.to_dict("records"),

        # Required so map callback can use selected row
        row_selectable="single",
        selected_rows=[0],

        # User-friendly interactive options
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        page_action="native",
        page_current=0,
        page_size=10,

        # Table styling
        style_table={
            "height": "330px",
            "overflowY": "auto",
            "overflowX": "auto",
            "border": "1px solid #cccccc"
        },
        style_cell={
            "textAlign": "left",
            "minWidth": "120px",
            "width": "150px",
            "maxWidth": "220px",
            "whiteSpace": "normal",
            "fontFamily": "Arial",
            "fontSize": "12px"
        },
        style_header={
            "fontWeight": "bold",
            "backgroundColor": "#f2f2f2"
        }
    ),

    html.Br(),
    html.Hr(),

    # Charts section
    html.Div(
        className="row",
        style={
            "display": "flex",
            "gap": "20px"
        },
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={
                    "width": "45%"
                }
            ),

            html.Div(
                id="map-id",
                className="col s12 m6",
                style={
                    "width": "55%"
                }
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Update the data table based on the selected rescue filter."""

    query = build_rescue_query(filter_type)
    filtered_df = get_dataframe_from_query(query)

    return filtered_df.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(view_data):
    """Update the breed pie chart based on the visible table data."""

    if view_data is None or len(view_data) == 0:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(view_data)

    if dff.empty or "breed" not in dff.columns:
        return [html.H4("No data available for the selected filter.")]

    # Show the most common breeds to keep the chart readable
    breed_counts = dff["breed"].value_counts().nlargest(10).reset_index()
    breed_counts.columns = ["breed", "count"]

    figure = px.pie(
        breed_counts,
        names="breed",
        values="count",
        title="Top Breeds in Selected Results"
    )

    return [
        dcc.Graph(figure=figure)
    ]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """Highlight selected columns in the data table."""

    if selected_columns is None:
        selected_columns = []

    return [
        {
            "if": {"column_id": column},
            "backgroundColor": "#D2F3FF"
        }
        for column in selected_columns
    ]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(view_data, selected_rows):
    """Update the geolocation map based on the selected table row."""

    if view_data is None or len(view_data) == 0:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(view_data)

    if dff.empty:
        return [html.H4("No map data available for the selected filter.")]

    if selected_rows is None or len(selected_rows) == 0:
        row = 0
    else:
        row = selected_rows[0]

    if row >= len(dff):
        row = 0

    # Safely read map values
    latitude = pd.to_numeric(dff.iloc[row]["location_lat"], errors="coerce")
    longitude = pd.to_numeric(dff.iloc[row]["location_long"], errors="coerce")
    breed = dff.iloc[row]["breed"]
    animal_name = dff.iloc[row]["name"]

    if pd.isna(latitude) or pd.isna(longitude):
        latitude = 30.75
        longitude = -97.48

    # Austin, TX is centered around [30.75, -97.48]
    return [
        dl.Map(
            style={
                "width": "100%",
                "height": "500px"
            },
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(animal_name)),
                            html.P("Breed: " + str(breed))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app.
# If this port is already in use, change 8055 to 8056 or 8057.
app.run_server(mode="external", port=8055, debug=False)

 * Running on http://127.0.0.1:8055/ (Press CTRL+C to quit)
127.0.0.1 - - [22/Jun/2026 00:19:05] "GET /_alive_949234fa-710c-494d-9d7c-4e4dfd8e649e HTTP/1.1" 200 -


Dash app running on https://aztecmailbox-cannonforum-3000.codio.io/proxy/8055/


127.0.0.1 - - [22/Jun/2026 00:19:28] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2026 00:19:28] "GET /_dash-component-suites/dash/deps/react-dom@16.v2_8_1m1752168217.14.0.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2026 00:19:28] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v2_8_1m1752168217.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2026 00:19:28] "GET /_dash-component-suites/dash/dash_table/bundle.v5_2_2m1752168217.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2026 00:19:29] "GET /_dash-component-suites/dash/deps/polyfill@7.v2_8_1m1752168217.12.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2026 00:19:29] "GET /_dash-component-suites/dash/html/dash_html_components.v2_0_8m1752168217.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2026 00:19:29] "GET /_dash-component-suites/dash/deps/prop-types@15.v2_8_1m1752168217.8.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Jun/2026 00:19:29] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v2_8_0m1752168217.js HTT